# SaccArbor: Diabetes Risk Prediction System
### MSc Advanced Machine Learning - Research Portfolio Project
**Author:** Roshan Perera  
**Institution:** MSc Advanced Machine Learning Mentorship Program  

This notebook implements a machine learning pipeline to predict diabetes risk using clinical measurements from the Pima Indians Diabetes Dataset.

## Import Libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix, 
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries imported. Random seed set to 42 for reproducibility.")

## Load Dataset

In [ ]:
data_path = "diabetes.csv"
if not os.path.exists(data_path):
    data_path = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"

df = pd.read_csv(data_path)
print(f"Dataset loaded: {df.shape[0]} patients, {df.shape[1]} features")
print("\nFeature data types:")
print(df.dtypes)

## Exploratory Data Analysis

In [ ]:
outcome_counts = df['Outcome'].value_counts()
print("Outcome distribution:")
print(outcome_counts)

df.describe().T

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Outcome', hue='Outcome', palette='Set2', legend=False)
plt.title("Distribution of Diabetic Outcomes (0 = Healthy, 1 = Diabetic)")
plt.xlabel("Clinical Status")
plt.ylabel("Patient Count")
plt.show()

print("Note: Moderate class imbalance (~65% non-diabetic, ~35% diabetic).")
print("Stratification will be used during train/test split.")

## Feature Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=df, x='Glucose', hue='Outcome', kde=True, palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Glucose Distribution by Outcome')
axes[0, 0].set_xlabel('Plasma Glucose (mg/dL)')
axes[0, 0].set_ylabel('Count')

sns.histplot(data=df, x='BMI', hue='Outcome', kde=True, palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('BMI Distribution by Outcome')
axes[0, 1].set_xlabel('Body Mass Index (kg/m²)')
axes[0, 1].set_ylabel('Count')

sns.histplot(data=df, x='Age', hue='Outcome', kde=True, palette='Set2', ax=axes[1, 0])
axes[1, 0].set_title('Age Distribution by Outcome')
axes[1, 0].set_xlabel('Age (Years)')
axes[1, 0].set_ylabel('Count')

sns.histplot(data=df, x='BloodPressure', hue='Outcome', kde=True, palette='Set2', ax=axes[1, 1])
axes[1, 1].set_title('Blood Pressure Distribution by Outcome')
axes[1, 1].set_xlabel('Diastolic Blood Pressure (mmHg)')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("Distribution plots generated for key clinical features.")

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap="Blues", fmt=".2f", cbar=True)
plt.title("Pearson Correlation Matrix of Patient Predictors")
plt.show()

print("Glucose and BMI show strongest correlation with Outcome.")
print("Age is strongly correlated with Pregnancies, as expected.")

## Data Cleaning & Imputation

## Feature Selection & Importance Analysis

Feature selection uses two approaches:
1. Correlation-based filtering to reduce multicollinearity
2. Random Forest feature importance ranking

In [ ]:
print("=== FEATURE SELECTION ANALYSIS ===\n")

print("1. Correlation Analysis with Target Variable:")
corr_with_target = df_clean.corr()['Outcome'].abs().sort_values(ascending=False)
print(corr_with_target)
print("\n")

corr_threshold = 0.1
selected_features_corr = corr_with_target[corr_with_target >= corr_threshold].index.tolist()
selected_features_corr.remove('Outcome')
print(f"Features with correlation >= {corr_threshold}: {selected_features_corr}")
print(f"Number of features selected: {len(selected_features_corr)}")
print("\n")

print("2. Random Forest Feature Importance Analysis:")
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(df_clean.drop(columns=['Outcome']), df_clean['Outcome'])

feature_importance = pd.DataFrame({
    'Feature': df_clean.drop(columns=['Outcome']).columns,
    'Importance': rf_selector.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance)
print("\n")

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='Importance', y='Feature', palette='viridis')
plt.title('Random Forest Feature Importance Rankings')
plt.xlabel('Importance Score')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()

importance_threshold = 0.05
selected_features_rf = feature_importance[feature_importance['Importance'] >= importance_threshold]['Feature'].tolist()
print(f"Features with importance >= {importance_threshold}: {selected_features_rf}")
print(f"Number of features selected: {len(selected_features_rf)}")
print("\n")

final_selected_features = list(set(selected_features_corr + selected_features_rf))
print(f"=== FINAL SELECTED FEATURES ===")
print(f"Total features: {final_selected_features}")
print(f"Number of features: {len(final_selected_features)}")

df_selected = df_clean[final_selected_features + ['Outcome']].copy()
print(f"\nFeature selection completed. Dataset reduced from {df_clean.shape[1]} to {df_selected.shape[1]} columns.")

In [ ]:
zero_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("Missing values (represented as 0s) per feature:")
for feat in zero_features:
    print(f"  - {feat}: {(df[feat] == 0).sum()} missing records")

df_clean = df.copy()
for col in zero_features:
    medians = df_clean[df_clean[col] > 0].groupby('Outcome')[col].median()
    df_clean.loc[(df_clean[col] == 0) & (df_clean['Outcome'] == 0), col] = medians[0]
    df_clean.loc[(df_clean[col] == 0) & (df_clean['Outcome'] == 1), col] = medians[1]

print("\nZero imputation completed. Total zeroes in cleaned dataset:")
print(df_clean[zero_features].eq(0).sum())

df_feat = df_selected.copy()

if 'Glucose' in final_selected_features and 'Age' in final_selected_features:
    df_feat['Glucose_Age_Interaction'] = df_feat['Glucose'] * df_feat['Age']

if 'BMI' in final_selected_features and 'Insulin' in final_selected_features:
    df_feat['BMI_Insulin_Ratio'] = df_feat['BMI'] / (df_feat['Insulin'] + 0.1)

if 'Age' in final_selected_features:
    df_feat['Age_Group'] = pd.cut(df_feat['Age'], bins=[20, 30, 45, 60, 100], labels=[0, 1, 2, 3]).astype(int)

print("Features engineered successfully. Enhanced Dataset columns:")
print(df_feat.columns.tolist())
print(f"\nFinal dataset dimensions: {df_feat.shape[0]} patients, {df_feat.shape[1]} features (including target)")

In [ ]:
df_feat = df_clean.copy()
df_feat['Glucose_Age_Interaction'] = df_feat['Glucose'] * df_feat['Age']
df_feat['BMI_Insulin_Ratio'] = df_feat['BMI'] / (df_feat['Insulin'] + 0.1)

df_feat['Age_Group'] = pd.cut(df_feat['Age'], bins=[20, 30, 45, 60, 100], labels=[0, 1, 2, 3]).astype(int)

print("Features engineered successfully. Enhanced Dataset columns:")
print(df_feat.columns.tolist())

X = df_feat.drop(columns=['Outcome'])
y = df_feat['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f"Training with {X_train_scaled.shape[1]} features after feature selection:")
print(f"Selected features: {X_train_scaled.columns.tolist()}")
print("\n")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest (Baseline)": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

print("Training baseline models...")
for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    print(f"  - {name}: Validation Accuracy = {acc:.4f}")

In [ ]:
X = df_feat.drop(columns=['Outcome'])
y = df_feat['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
scale_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 
              'DiabetesPedigreeFunction', 'Age', 'Glucose_Age_Interaction', 'BMI_Insulin_Ratio']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest (Baseline)": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

print("Training baseline models...")
for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    print(f"  - {name}: Validation Accuracy = {acc:.4f}")

## Confusion Matrices for All Models

In [ ]:
print("=== CONFUSION MATRICES FOR ALL MODELS ===\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

model_predictions = {}
for idx, (name, clf) in enumerate(models.items()):
    y_pred = clf.predict(X_test_scaled)
    model_predictions[name] = y_pred
    
    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["Healthy", "Diabetic"],
        cmap="Blues",
        ax=axes[idx],
        colorbar=False
    )
    axes[idx].set_title(f'{name}')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

if len(models) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig('confusion_matrices_all_models.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrices generated and saved as 'confusion_matrices_all_models.png'")
print("\n")

print("=== DETAILED CONFUSION MATRIX METRICS ===\n")
for name, y_pred in model_predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"{name}:")
    print(f"  True Negatives: {tn}")
    print(f"  False Positives: {fp}")
    print(f"  False Negatives: {fn}")
    print(f"  True Positives: {tp}")
    print(f"  Sensitivity (Recall): {tp/(tp+fn):.4f}")
    print(f"  Specificity: {tn/(tn+fp):.4f}")
    print()

## Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_rf = grid_search.best_estimator_
print(f"Optimal parameters: {grid_search.best_params_}")
print(f"Best tuned RF accuracy: {accuracy_score(y_test, best_rf.predict(X_test_scaled)):.4f}")

## Model Evaluation

In [ ]:
tuned_preds = best_rf.predict(X_test_scaled)
print("Classification report:")
print(classification_report(y_test, tuned_preds))

cm = confusion_matrix(y_test, tuned_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Healthy", "Diabetic"], yticklabels=["Healthy", "Diabetic"])
plt.ylabel('Actual Clinical State')
plt.xlabel('Predicted Diagnostic Boundary')
plt.title('Confusion Matrix - Tuned Random Forest')
plt.show()

In [ ]:
probs = best_rf.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, probs)
auc_score = roc_auc_score(y_test, probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Random Forest ROC (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

## Model Saving

In [ ]:
joblib.dump(best_rf, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Model and scaler saved to disk.")

## Clinical Diagnostic Examples

In [ ]:
def evaluate_patient(patient_profile):
    patient_df = pd.DataFrame([patient_profile])
    
    patient_df['Glucose_Age_Interaction'] = patient_df['Glucose'] * patient_df['Age']
    patient_df['BMI_Insulin_Ratio'] = patient_df['BMI'] / (patient_df['Insulin'] + 0.1)
    
    bins = [20, 30, 45, 60, 100]
    patient_df['Age_Group'] = pd.cut(patient_df['Age'], bins=bins, labels=[0, 1, 2, 3]).astype(int)
    
    scaled_patient = patient_df.copy()
    scaled_patient[scale_cols] = scaler.transform(patient_df[scale_cols])
    
    prob = best_rf.predict_proba(scaled_patient)[:, 1][0]
    pred = best_rf.predict(scaled_patient)[0]
    
    print(f"Patient Age: {patient_profile['Age']} | Glucose: {patient_profile['Glucose']} mg/dL | BMI: {patient_profile['BMI']} kg/m²")
    print(f"Risk Probability: {prob*100:.1f}% -> Classification: {'Diabetic (1)' if pred == 1 else 'Healthy (0)'}\n")

high_risk_patient = {
    'Pregnancies': 5, 'Glucose': 168, 'BloodPressure': 82, 'SkinThickness': 35,
    'Insulin': 140, 'BMI': 38.6, 'DiabetesPedigreeFunction': 0.85, 'Age': 48
}

low_risk_patient = {
    'Pregnancies': 1, 'Glucose': 95, 'BloodPressure': 66, 'SkinThickness': 18,
    'Insulin': 60, 'BMI': 24.2, 'DiabetesPedigreeFunction': 0.22, 'Age': 24
}

print("Patient assessment tests:")
evaluate_patient(high_risk_patient)
evaluate_patient(low_risk_patient)